# Séance 11 · Le RAG : donner de la mémoire à son IA · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

Un LLM a lu tout internet, mais il n'a jamais lu **tes** notes de cours, les règles de ton jeu ou ton livre préféré. Aujourd'hui on lui donne cette mémoire avec le **RAG** (*Retrieval-Augmented Generation*) : on cherche les bons passages dans tes documents et on les glisse dans le prompt au bon moment.

Tout tourne dans **Google Colab** (menu *Exécution → Modifier le type d'exécution → T4 GPU*). Exécute chaque cellule avec `Maj + Entrée`.

**Livrable de la séance** : un assistant RAG qui répond à des questions sur des documents que tu as choisis (tes notes, les règles d'un jeu, un résumé de livre).


## Préparation

La cellule `llm(messages)` des séances précédentes, plus les bibliothèques pour les vecteurs.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import numpy as np
import matplotlib.pyplot as plt

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Mode démo : sans contexte, le faux modèle invente ; avec contexte, il recopie le passage le plus utile."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "Contexte :" in systeme:                                   # un RAG lui a donné des passages
        contexte = systeme.split("Contexte :", 1)[1]
        mots_question = {m for m in re.findall(r"\w{4,}", ql)} - {"quel", "quelle", "quels", "comment", "combien", "pourquoi", "dans", "avec", "pour"}
        phrases = [p.strip(" -\n") for p in re.split(r"(?<=[.!?])\s+", contexte) if p.strip(" -\n")]
        meilleure = max(phrases, key=lambda p: sum(m in p.lower() for m in mots_question), default="")
        if meilleure and sum(m in meilleure.lower() for m in mots_question) > 0:
            return "D'après tes documents : " + meilleure
        return "Je ne trouve pas cette information dans les documents fournis."
    if "sardine" in ql:
        return "Sardine Express est un jeu de cartes rapide où chaque joueur doit se débarrasser de ses sardines avant les autres. Il se joue avec 52 cartes classiques."
    if "prof" in ql or "cours" in ql:
        return "Le cours a lieu le lundi matin, dans la salle 12, avec le professeur Dupont."
    return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible : c'est un sujet intéressant."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

## 1. Le problème : le modèle ne connaît pas mes documents

On va inventer un jeu de cartes, *Sardine Express*, et écrire ses règles nous-mêmes. Aucun modèle au monde ne les connaît. Regarde ce qu'il répond quand on lui pose une question dessus : il ne dit pas « je ne sais pas », il **invente** (c'est l'hallucination de la séance 9).

In [ ]:
def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

print(demander("Dans le jeu Sardine Express, que se passe-t-il quand on pose une carte Mouette ?"))

La solution est simple : c'est un **examen à livre ouvert**. Au lieu de demander au modèle de se souvenir, on lui tend la bonne page au moment de la question. Tout le RAG consiste à trouver **la bonne page**.

Voici les règles du jeu (15 paragraphes). Lis-les en diagonale : c'est notre « livre ».

In [ ]:
REGLES_DU_JEU = """Sardine Express est un jeu de cartes pour 2 à 5 joueurs, à partir de 8 ans. Une partie dure environ 15 minutes.

Le jeu contient 60 cartes Sardine (numérotées de 1 à 12, en 5 couleurs), 8 cartes Mouette et 4 cartes Tempête. On utilise aussi un dé à six faces.

Au début de la partie, on mélange toutes les cartes. Chaque joueur reçoit 7 cartes, et le reste forme la pioche, face cachée. La carte du dessus de la pioche est retournée : c'est le début de la défausse.

Le joueur le plus jeune commence. Ensuite, on joue dans le sens des aiguilles d'une montre.

À son tour, un joueur doit poser une carte Sardine de la même couleur OU du même numéro que la carte du dessus de la défausse. S'il ne peut pas, il pioche une carte et son tour est terminé.

La carte Mouette peut être posée sur n'importe quelle carte. Le joueur suivant doit alors piocher 2 cartes et passer son tour. Deux Mouettes ne peuvent pas être posées l'une sur l'autre.

La carte Tempête inverse le sens du jeu. Si elle est posée, on lance le dé : sur un 6, tous les joueurs passent leur main entière à leur voisin de gauche.

Quand un joueur n'a plus qu'une seule carte en main, il doit crier « Sardine ! ». S'il oublie et qu'un autre joueur le remarque avant le tour suivant, il pioche 3 cartes de pénalité.

Le premier joueur qui n'a plus de cartes gagne la manche. Il marque 1 point par carte restante dans la main de chaque adversaire, et 5 points par Mouette restante.

Une partie complète se joue en 3 manches. Le joueur avec le plus de points à la fin des 3 manches gagne la partie. En cas d'égalité, on joue une manche de plus.

Variante « Banc de sardines » : si un joueur a en main 3 cartes du même numéro, il peut les poser d'un coup à son tour, quelle que soit la couleur.

Variante « Mode rapide » : chaque joueur reçoit 5 cartes au lieu de 7, et la partie se joue en une seule manche.

Si la pioche est vide, on mélange la défausse (sauf la carte du dessus) pour former une nouvelle pioche.

Il est interdit de regarder les cartes des autres joueurs. Un joueur surpris à tricher perd immédiatement la manche en cours.

Le jeu a été créé en 2019 par Léa Marchand et illustré par Tom Ravel. Il est édité par les éditions du Phare, à Brest."""

print(REGLES_DU_JEU[:400], "...")
print("\nLongueur :", len(REGLES_DU_JEU), "caractères,", len(REGLES_DU_JEU.split()), "mots")

**Exercice** : donne au modèle **tout** le texte des règles dans le prompt système, puis repose la question sur la carte Mouette. Ça marche ? Alors pourquoi ne pas toujours faire ça ? (Pense au coût en tokens de la séance 9, et à un livre de 300 pages.)

In [ ]:
# À toi
systeme = "Réponds en français à partir du texte suivant.\nContexte : " + REGLES_DU_JEU
print(demander("Que se passe-t-il quand on pose une carte Mouette ?", systeme=systeme))

<details><summary>Solution</summary>

```python
# Ça marche : le modèle a la réponse sous les yeux. Mais un petit modèle a une fenêtre de quelques milliers
# de tokens, et chaque token coûte (temps ou argent). Un livre entier ne rentre pas, et même s'il rentrait,
# le modèle se perd dans trop de texte. D'où l'idée : ne donner QUE les passages utiles. C'est le RAG.
```

</details>

## 2. Les embeddings : des mots proches sont des points proches

Pour trouver « le bon passage », il faut mesurer si deux textes parlent de la même chose. L'astuce s'appelle **embedding** : transformer un mot ou une phrase en une liste de nombres (un **vecteur**), de façon que les textes au sens proche donnent des points proches.

Pour voir l'idée, on écrit à la main des vecteurs en 2 dimensions (dans la vraie vie il y en a 384 ou 1 536, mais on ne peut pas les dessiner).

In [ ]:
# Deux nombres par mot : (animal ↔ objet, petit ↔ gros). Écrits à la main !
mots = {
    "chat": (0.9, 0.2), "chien": (0.9, 0.4), "lion": (0.95, 0.9), "souris": (0.85, 0.05),
    "pomme": (0.3, 0.1), "banane": (0.3, 0.15), "pastèque": (0.35, 0.5),
    "vélo": (0.05, 0.4), "voiture": (0.05, 0.7), "camion": (0.02, 0.95),
}

plt.figure(figsize=(6, 5))
for mot, (x, y) in mots.items():
    plt.scatter(x, y)
    plt.annotate(mot, (x + 0.01, y + 0.01))
plt.xlabel("objet  ←→  animal")
plt.ylabel("petit  ←→  gros")
plt.title("Des mots proches sont des points proches")
plt.grid(alpha=0.3)
plt.show()

Pour mesurer la proximité, on utilise la **similarité cosinus** : 1 = même direction (même sens), 0 = rien à voir. Elle regarde l'angle entre deux vecteurs, pas leur longueur.

In [ ]:
def similarite(a, b):
    """Similarité cosinus entre deux vecteurs (listes de nombres)."""
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("chat  vs chien  :", round(similarite(mots["chat"], mots["chien"]), 3))
print("chat  vs camion :", round(similarite(mots["chat"], mots["camion"]), 3))
print("pomme vs banane :", round(similarite(mots["pomme"], mots["banane"]), 3))

**Exercice** : ajoute le mot `"hamster"` avec des coordonnées de ton choix, puis écris une boucle qui trouve le mot du dictionnaire le plus proche de lui.

In [ ]:
# À toi
mots["hamster"] = (0.9, 0.03)

<details><summary>Solution</summary>

```python
mots["hamster"] = (0.9, 0.03)
meilleur = max((m for m in mots if m != "hamster"), key=lambda m: similarite(mots["hamster"], mots[m]))
print("Le plus proche de hamster :", meilleur)   # souris
```

</details>

## 3. Étape 1 : découper les documents (chunks)

Un RAG a 5 étapes : **découper, vectoriser, chercher, injecter, répondre**. On les fait une par une.

On ne vectorise pas un livre entier d'un coup : on le découpe en **chunks** (morceaux) de quelques phrases, chacun sur une idée. Ici, un paragraphe = un chunk, c'est le découpage le plus simple.

In [ ]:
def decouper(texte):
    """Un paragraphe (séparé par une ligne vide) = un chunk."""
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

chunks = decouper(REGLES_DU_JEU)
print(len(chunks), "chunks")
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] ({len(c.split())} mots) {c[:80]}...")

**Exercice** : certains textes n'ont pas de paragraphes. Écris `decouper_par_taille(texte, taille=40)` qui coupe le texte en morceaux d'environ 40 mots, en coupant de préférence à la fin d'une phrase (après un point).

In [ ]:
# À toi
def decouper_par_taille(texte, taille=40):
    phrases = re.split(r"(?<=[.!?])\s+", texte.replace("\n", " "))
    morceaux, courant = [], ""
    for p in phrases:
        courant += p + " "
        # ... quand `courant` dépasse `taille` mots, on l'ajoute à `morceaux` et on repart de zéro
    if courant.strip():
        morceaux.append(courant.strip())
    return morceaux

print(len(decouper_par_taille(REGLES_DU_JEU)), "morceaux")

<details><summary>Solution</summary>

```python
def decouper_par_taille(texte, taille=40):
    phrases = re.split(r"(?<=[.!?])\s+", texte.replace("\n", " "))
    morceaux, courant = [], ""
    for p in phrases:
        courant += p + " "
        if len(courant.split()) >= taille:
            morceaux.append(courant.strip())
            courant = ""
    if courant.strip():
        morceaux.append(courant.strip())
    return morceaux

for m in decouper_par_taille(REGLES_DU_JEU)[:3]:
    print(len(m.split()), "mots :", m[:60], "...")
```

</details>

## 4. Étape 2 : vectoriser

Chaque chunk devient un vecteur. Deux façons :
- **sentence-transformers** : un petit modèle multilingue qui comprend le sens (« carte qui fait piocher » ≈ « Mouette »). C'est ce qu'on utilise si l'installation marche.
- **TF-IDF** (scikit-learn) : compte les mots importants. Pas de sens, mais ça marche sans modèle : notre **repli**.

On enveloppe les deux dans la même fonction `vectoriser(textes)`, comme ça la suite du notebook ne change pas.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# mots trop fréquents pour aider la recherche ("le", "une", "quand"...) : TF-IDF les ignore
STOP_FR = ("le la les l un une des du de d et ou à a au aux en dans sur par pour avec sans ce cet cette ces se son sa ses "
           "leur leurs il elle ils elles on ne pas plus que qui quoi quel quelle quels dont où quand comment est sont être avoir fait y t").split()

if USE_MODEL:
    %pip install -q sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    encodeur = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    METHODE = "sentence-transformers"
except Exception as e:
    METHODE = "tfidf"
    tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)      # le vocabulaire est appris sur nos chunks
print("Méthode d'embedding :", METHODE)

def vectoriser(textes):
    """Liste de textes → tableau numpy (une ligne = un vecteur)."""
    if METHODE == "sentence-transformers":
        return encodeur.encode(textes)
    return tfidf.transform(textes).toarray()

vecteurs = vectoriser(chunks)
print("Forme du tableau :", vecteurs.shape, "→", len(chunks), "chunks,", vecteurs.shape[1], "dimensions chacun")

**Exercice** : vectorise les phrases « la carte qui fait piocher deux cartes » et « la carte Mouette ». Avec `similarite`, sont-elles proches ? Compare avec « le jeu a été créé en 2019 ». Si tu es en TF-IDF, pourquoi le score est-il si bas ?

In [ ]:
# À toi
a, b, c = vectoriser(["la carte qui fait piocher deux cartes", "la carte Mouette", "le jeu a été créé en 2019"])
print("piocher vs Mouette :", round(similarite(a, b), 3))
print("piocher vs 2019    :", round(similarite(a, c), 3))

<details><summary>Solution</summary>

```python
# Avec sentence-transformers, "piocher deux cartes" et "Mouette" sont proches : le modèle a compris le sens.
# Avec TF-IDF, seul le mot "carte" est en commun : score faible. TF-IDF ne connaît pas le sens, seulement les mots.
# C'est la différence entre chercher par mots-clés (Ctrl+F) et chercher par idée.
```

</details>

## 5. Étape 3 : chercher les passages pertinents

La question devient un vecteur à son tour, et on garde les `k` chunks dont le vecteur est le plus proche. C'est notre **moteur de recherche** : il renvoie des passages, pas encore une réponse.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def chercher(question, k=3):
    """Renvoie les k chunks les plus proches de la question, avec leur score."""
    q = vectoriser([question])
    scores = cosine_similarity(q, vecteurs)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [(chunks[i], round(float(scores[i]), 3)) for i in meilleurs]

for passage, score in chercher("Que fait la carte Mouette ?"):
    print(f"[{score}] {passage[:90]}...")

**Exercice** : teste `chercher` avec une question dont la réponse **n'est pas** dans les règles (« Qui a gagné la Coupe du monde 2022 ? »). Que valent les scores ? Propose un seuil en dessous duquel on considère qu'on n'a rien trouvé.

In [ ]:
# À toi
for passage, score in chercher("Qui a gagné la Coupe du monde 2022 ?"):
    print(f"[{score}] {passage[:90]}...")

<details><summary>Solution</summary>

```python
# Les scores sont bien plus bas que pour une vraie question sur le jeu (souvent < 0,2 en TF-IDF).
# On peut fixer un seuil : si le meilleur score est sous 0,15, on répond "je n'ai rien trouvé dans les documents".
SEUIL = 0.15
def chercher_avec_seuil(question, k=3):
    resultats = chercher(question, k)
    return [r for r in resultats if r[1] >= SEUIL]
print(chercher_avec_seuil("Qui a gagné la Coupe du monde 2022 ?"))
```

</details>

## 6. Étapes 4 et 5 : injecter dans le prompt et répondre

Dernière étape : on colle les passages trouvés dans le prompt système (**injecter**), avec la consigne de ne répondre **qu'à partir d'eux**, puis on pose la question (**répondre**). Regarde bien le prompt construit : c'est tout le secret du RAG.

In [ ]:
def rag(question, k=3, afficher_prompt=False):
    passages = [p for p, score in chercher(question, k)]
    contexte = "\n".join(f"- {p}" for p in passages)
    systeme = ("Tu réponds en français, en 2 phrases maximum, UNIQUEMENT à partir du contexte ci-dessous. "
               "Si la réponse n'y est pas, dis : je ne trouve pas cette information dans les documents.\n"
               f"Contexte :\n{contexte}")
    if afficher_prompt:
        print("=== PROMPT SYSTÈME ===\n" + systeme + "\n======================")
    return demander(question, systeme=systeme)

print(rag("Que se passe-t-il quand on pose une carte Mouette ?", afficher_prompt=True))

In [ ]:
questions = [
    "Combien de cartes reçoit chaque joueur au début ?",
    "Qui a créé le jeu ?",
    "Que se passe-t-il si on oublie de crier Sardine ?",
]
for q in questions:
    print("Q :", q)
    print("   sans RAG :", demander(q))
    print("   avec RAG :", rag(q), "\n")

**Exercice** : pose à `rag` une question hors sujet (« Quelle est la capitale du Japon ? »). Le modèle respecte-t-il la consigne « je ne trouve pas » ? Puis pose une question **piège** dont la réponse est dans les règles mais avec d'autres mots (« combien de temps dure une partie ? » alors que le texte dit « 15 minutes »).

In [ ]:
# À toi
print(rag("Quelle est la capitale du Japon ?"))
print(rag("Combien de temps dure une partie ?"))

<details><summary>Solution</summary>

```python
# Hors sujet : un gros modèle dit "je ne trouve pas" ; un petit modèle répond parfois quand même (hallucination).
# Question avec d'autres mots : sentence-transformers la relie à "15 minutes" (sens) ; TF-IDF y arrive
# seulement si un mot est en commun ("partie"). C'est pour ça que les vrais RAG utilisent des embeddings sémantiques.
```

</details>

## 7. Projet : « pose une question à ton cours » (80 min)

Tu as toutes les briques. Construis maintenant **ton** assistant sur **tes** documents : tes notes de cours, les règles d'un jeu que tu aimes, le résumé d'un livre ou d'une série. Par défaut, `MES_NOTES` contient un résumé des séances 9 à 11 : remplace-le par ton texte (au moins 8 paragraphes, séparés par une ligne vide).

Étapes : (1) colle ton texte, (2) découpe et vectorise, (3) pose 5 questions et note si la réponse est bonne, (4) améliore (taille des chunks, `k`, consigne du prompt).

In [ ]:
# Question 1 : colle ton texte ici (paragraphes séparés par une ligne vide)
MES_NOTES = """Séance 9 : un token est un morceau de mot transformé en nombre. Le modèle ne voit pas les lettres, c'est pour ça qu'il compte mal les r de strawberry.

Séance 9 : un LLM fait une seule chose, prédire le token suivant, encore et encore. La température règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.

Séance 9 : un LLM hallucine, c'est-à-dire qu'il invente une réponse plausible quand il ne sait pas. Il a une date de connaissance et il calcule mal.

Séance 10 : une API, c'est comme un serveur de restaurant. On envoie une commande (la requête) et on reçoit un plat (la réponse), souvent en JSON.

Séance 10 : les trois rôles sont system (les consignes), user (l'utilisateur) et assistant (le modèle). Le prompt système donne la personnalité et les règles.

Séance 10 : un chatbot n'a pas de mémoire, il renvoie tout l'historique des messages à chaque tour. Pour réutiliser une réponse dans un programme, on demande du JSON.

Séance 11 : le RAG donne au modèle les bons passages de mes documents avant de poser la question. Les étapes sont découper, vectoriser, chercher, injecter, répondre."""

mes_chunks = decouper(MES_NOTES)
print(len(mes_chunks), "chunks dans mes notes")

In [ ]:
# Question 2 : construis l'index (vectorise) : on refait les 3 fonctions pour TES notes
if METHODE == "tfidf":
    tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(mes_chunks)      # nouveau vocabulaire pour tes notes
mes_vecteurs = vectoriser(mes_chunks)

def chercher_notes(question, k=2):
    scores = cosine_similarity(vectoriser([question]), mes_vecteurs)[0]
    return [(mes_chunks[i], round(float(scores[i]), 3)) for i in scores.argsort()[::-1][:k]]

def mon_assistant(question, k=2):
    contexte = "\n".join(f"- {p}" for p, s in chercher_notes(question, k))
    systeme = ("Tu es l'assistant de révision de l'élève. Tu réponds en français, en 2 phrases maximum, "
               "UNIQUEMENT à partir du contexte. Si la réponse n'y est pas, dis : je ne trouve pas cette information dans les documents.\n"
               f"Contexte :\n{contexte}")
    return demander(question, systeme=systeme)

print(mon_assistant("C'est quoi un token ?"))

In [ ]:
# Question 3 : pose 5 questions et évalue (True si la réponse est correcte)
mes_questions = [
    "C'est quoi un token ?",
    "À quoi sert la température ?",
    "Quels sont les trois rôles dans une conversation ?",
    "Quelles sont les étapes du RAG ?",
    "Qui a gagné la Coupe du monde 2022 ?",      # hors sujet : il doit dire qu'il ne trouve pas
]
evaluation = []
for q in mes_questions:
    reponse = mon_assistant(q)
    print("Q :", q)
    print("R :", reponse, "\n")
    evaluation.append({"question": q, "reponse": reponse, "correcte": None})   # ← remplis True / False

In [ ]:
# Question 4 : ta fiche de résultats (remplis "correcte" ci-dessus, puis lance cette cellule)
for e in evaluation:
    e["correcte"] = e["correcte"] if e["correcte"] is not None else "?"
bonnes = sum(1 for e in evaluation if e["correcte"] is True)
print(f"=== MON ASSISTANT RAG ===")
print(f"Documents : {len(mes_chunks)} chunks | embeddings : {METHODE} | k = 2")
print(f"Score : {bonnes} / {len(evaluation)} bonnes réponses")
for e in evaluation:
    print(f"  [{e['correcte']}] {e['question']}")

**Pour aller plus loin** : (a) change `k` (1, 2, 5) et regarde l'effet ; (b) essaie `decouper_par_taille` à la place de `decouper` ; (c) ajoute un seuil de score pour refuser les questions hors sujet **avant** d'appeler le modèle (ça économise des tokens !).

## À retenir

- Un LLM ne connaît pas tes documents. Si on ne lui donne rien, il **invente**.
- **RAG** = examen à livre ouvert : on cherche les bons passages et on les glisse dans le prompt au moment de la question.
- Un **embedding** transforme un texte en vecteur ; des textes au sens proche donnent des points proches. La **similarité cosinus** mesure cette proximité.
- Les 5 étapes : **découper** (chunks), **vectoriser**, **chercher** (les k plus proches), **injecter** (dans le prompt système), **répondre**.
- TF-IDF cherche par mots-clés ; sentence-transformers cherche par sens. Les vrais RAG utilisent le sens.
- La consigne « réponds uniquement à partir du contexte, sinon dis que tu ne sais pas » est ce qui limite les hallucinations.
- Un RAG se règle : taille des chunks, nombre `k`, seuil de score, consigne du prompt.

## Pour montrer aux autres

Présente ton assistant en 2 minutes :
1. Quels documents lui as-tu donnés, et pourquoi ceux-là ?
2. Montre une question où il répond juste, et une où il se trompe : que s'est-il passé dans la recherche (affiche les passages trouvés) ?
3. Qu'est-ce qui changerait avec un gros modèle et des vrais embeddings ?

Liens gratuits
- Modèle d'embeddings multilingue utilisé : https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Visualiser des embeddings en 3D : https://projector.tensorflow.org
- NotebookLM (un RAG prêt à l'emploi de Google, pour comparer) : https://notebooklm.google
- scikit-learn TF-IDF : https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting